In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
import os
from torchvision.utils import make_grid
import torchvision

# Set random seed for reproducibility
torch.manual_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load dataset (MNIST)
transform = transforms.Compose([transforms.ToTensor()])
full_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)

# Split dataset into train, validation, and test sets (80%, 10%, 10%)
train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
train_dataset, val_test_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size + test_size])
val_dataset, test_dataset = torch.utils.data.random_split(val_test_dataset, [val_size, test_size])

# Create data loaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Dataset sizes - Train: {len(train_dataset)}, Validation: {len(val_dataset)}, Test: {len(test_dataset)}")

# Create directories for saving model checkpoints and generated images
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('generated_images', exist_ok=True)

# Define the Simple Autoencoder architecture
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=20):
        super(Autoencoder, self).__init__()

        # Encoder layers
        self.encoder = nn.Sequential(
            nn.Linear(28 * 28, 512),  # Input image size for MNIST is 28x28
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim)  # Bottleneck layer (latent space)
        )

        # Decoder layers
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 28 * 28),
            nn.Sigmoid()  # Output values between 0 and 1 for image reconstruction
        )

    def forward(self, x):
        # Flatten the input image
        x = x.view(x.size(0), -1)

        # Encode input to latent representation
        latent = self.encoder(x)

        # Decode from latent space to reconstruction
        reconstruction = self.decoder(latent)

        # Reshape back to image dimensions
        reconstruction = reconstruction.view(x.size(0), 1, 28, 28)

        return reconstruction, latent

    def encode(self, x):
        # Encode input to latent representation (for visualization)
        x = x.view(x.size(0), -1)
        return self.encoder(x)

# Define the Variational Autoencoder (VAE) architecture
class VAE(nn.Module):
    def __init__(self, latent_dim=20):
        super(VAE, self).__init__()

        # Encoder layers
        self.encoder_layers = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU()
        )

        # Mean and log variance layers for the latent distribution
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

        # Decoder layers
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 28 * 28),
            nn.Sigmoid()
        )

        self.latent_dim = latent_dim

    def encode(self, x):
        # Flatten the input image
        x = x.view(x.size(0), -1)

        # Pass through encoder layers
        x = self.encoder_layers(x)

        # Get mean and log variance for latent distribution
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)

        return mu, logvar

    def reparameterize(self, mu, logvar):
        # Reparameterization trick: z = mu + std * eps
        # where eps is a random noise from standard normal distribution
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z

    def decode(self, z):
        # Decode from latent space to reconstruction
        reconstruction = self.decoder(z)
        reconstruction = reconstruction.view(-1, 1, 28, 28)
        return reconstruction

    def forward(self, x):
        # Encode input to latent distribution parameters
        mu, logvar = self.encode(x)

        # Sample from the latent distribution using reparameterization trick
        z = self.reparameterize(mu, logvar)

        # Decode from latent space to reconstruction
        reconstruction = self.decode(z)

        return reconstruction, mu, logvar

# Loss function for VAE
def vae_loss(reconstruction, x, mu, logvar, beta=1.0):
    # Flatten the input image for comparison
    x_flat = x.view(x.size(0), -1)
    recon_flat = reconstruction.view(reconstruction.size(0), -1)

    # Reconstruction loss (binary cross entropy)
    recon_loss = F.binary_cross_entropy(recon_flat, x_flat, reduction='sum')

    # KL divergence loss
    # KL(N(mu, var) || N(0, 1)) = 0.5 * sum(1 + log(var) - mu^2 - var)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    # Total loss with beta weighting for KL divergence term
    total_loss = recon_loss + beta * kl_loss

    return total_loss

# Function for training both types of autoencoders
def train_model(model, train_loader, val_loader, optimizer, num_epochs=10,
                is_vae=False, early_stopping_patience=5, checkpoint_path=None):
    best_val_loss = float('inf')
    patience_counter = 0
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0

        for batch_idx, (data, _) in enumerate(train_loader):
            data = data.to(device)
            optimizer.zero_grad()

            if is_vae:
                reconstruction, mu, logvar = model(data)
                loss = vae_loss(reconstruction, data, mu, logvar)
            else:
                reconstruction, _ = model(data)
                loss = F.mse_loss(reconstruction, data, reduction='sum')

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            if batch_idx % 100 == 0:
                print(f"Epoch: {epoch+1}/{num_epochs}, Batch: {batch_idx}/{len(train_loader)}, Loss: {loss.item()/len(data):.6f}")

        avg_train_loss = train_loss / len(train_loader.dataset)
        train_losses.append(avg_train_loss)

        # Validation phase
        model.eval()
        val_loss = 0

        with torch.no_grad():
            for data, _ in val_loader:
                data = data.to(device)

                if is_vae:
                    reconstruction, mu, logvar = model(data)
                    loss = vae_loss(reconstruction, data, mu, logvar)
                else:
                    reconstruction, _ = model(data)
                    loss = F.mse_loss(reconstruction, data, reduction='sum')

                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader.dataset)
        val_losses.append(avg_val_loss)

        print(f"Epoch: {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.6f}, Validation Loss: {avg_val_loss:.6f}")

        # Save checkpoint if validation loss improves
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0

            if checkpoint_path:
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': best_val_loss,
                }, checkpoint_path)
                print(f"Checkpoint saved to {checkpoint_path}")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= early_stopping_patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break

    # Load best model from checkpoint
    if checkpoint_path and os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Loaded best model from checkpoint (epoch {checkpoint['epoch']+1})")

    return model, train_losses, val_losses

# Function to visualize latent space
def visualize_latent_space(model, test_loader, num_samples=1000, is_vae=False):
    model.eval()

    # Get latent representations and labels for test samples
    latent_vectors = []
    labels = []

    with torch.no_grad():
        for i, (data, target) in enumerate(test_loader):
            if len(latent_vectors) * test_loader.batch_size >= num_samples:
                break

            data = data.to(device)

            if is_vae:
                mu, _ = model.encode(data)
                latent_vectors.append(mu.cpu().numpy())
            else:
                _, latent = model(data)
                latent_vectors.append(latent.cpu().numpy())

            labels.append(target.numpy())

    latent_vectors = np.vstack(latent_vectors)[:num_samples]
    labels = np.concatenate(labels)[:num_samples]

    # Perform PCA or t-SNE if latent dimension > 2
    if latent_vectors.shape[1] > 2:
        from sklearn.decomposition import PCA
        pca = PCA(n_components=2)
        latent_vectors_2d = pca.fit_transform(latent_vectors)
        title_suffix = " (PCA)"
    else:
        latent_vectors_2d = latent_vectors
        title_suffix = ""

    # Create a scatter plot of the 2D latent space
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(latent_vectors_2d[:, 0], latent_vectors_2d[:, 1], c=labels, cmap='tab10', alpha=0.6)
    plt.colorbar(scatter, label='Digit')
    model_type = "VAE" if is_vae else "Autoencoder"
    plt.title(f"{model_type} Latent Space Visualization{title_suffix}")
    plt.xlabel("Latent Dimension 1")
    plt.ylabel("Latent Dimension 2")
    plt.grid(True, alpha=0.3)
    return plt

# Function to visualize reconstructions
def visualize_reconstructions(model, test_loader, num_samples=10, is_vae=False):
    model.eval()

    # Get original and reconstructed samples
    originals = []
    reconstructions = []

    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(device)

            if is_vae:
                recon, _, _ = model(data)
            else:
                recon, _ = model(data)

            originals.append(data.cpu().numpy())
            reconstructions.append(recon.cpu().numpy())

            if len(originals) * test_loader.batch_size >= num_samples:
                break

    # Flatten the lists
    originals = np.vstack(originals)[:num_samples]
    reconstructions = np.vstack(reconstructions)[:num_samples]

    # Plot original and reconstructed images side by side
    fig, axes = plt.subplots(2, num_samples, figsize=(15, 4))
    model_type = "VAE" if is_vae else "Autoencoder"
    fig.suptitle(f"{model_type} Reconstructions", fontsize=16)

    for i in range(num_samples):
        # Original images
        axes[0, i].imshow(originals[i, 0], cmap='gray')
        axes[0, i].set_xticks([])
        axes[0, i].set_yticks([])
        if i == 0:
            axes[0, i].set_ylabel('Original')

        # Reconstructed images
        axes[1, i].imshow(reconstructions[i, 0], cmap='gray')
        axes[1, i].set_xticks([])
        axes[1, i].set_yticks([])
        if i == 0:
            axes[1, i].set_ylabel('Reconstructed')

    plt.tight_layout()
    plt.subplots_adjust(top=0.85)
    return plt

# Function to test model performance
def test_model(model, test_loader, is_vae=False):
    model.eval()
    test_loss = 0

    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(device)

            if is_vae:
                reconstruction, mu, logvar = model(data)
                loss = vae_loss(reconstruction, data, mu, logvar)
            else:
                reconstruction, _ = model(data)
                loss = F.mse_loss(reconstruction, data, reduction='sum')

            test_loss += loss.item()

    avg_test_loss = test_loss / len(test_loader.dataset)
    model_type = "VAE" if is_vae else "Autoencoder"
    print(f"{model_type} Test Loss: {avg_test_loss:.6f}")
    return avg_test_loss

# Main execution
if __name__ == "__main__":
    # 1. Train and evaluate the Simple Autoencoder
    print("\n=== Simple Autoencoder ===")
    latent_dim = 20

    # Initialize model and optimizer
    autoencoder = Autoencoder(latent_dim=latent_dim).to(device)
    ae_optimizer = optim.Adam(autoencoder.parameters(), lr=1e-3)

    # Train the model
    autoencoder, ae_train_losses, ae_val_losses = train_model(
        autoencoder, train_loader, val_loader, ae_optimizer,
        num_epochs=20, is_vae=False, early_stopping_patience=5,
        checkpoint_path='checkpoints/autoencoder.pt'
    )

    # Test the model
    ae_test_loss = test_model(autoencoder, test_loader, is_vae=False)

    # Visualize latent space
    ae_latent_plot = visualize_latent_space(autoencoder, test_loader, is_vae=False)
    ae_latent_plot.savefig('autoencoder_latent_space.png')
    plt.close()

    # Visualize reconstructions
    ae_recon_plot = visualize_reconstructions(autoencoder, test_loader, is_vae=False)
    ae_recon_plot.savefig('autoencoder_reconstructions.png')
    plt.close()

    # Plot training and validation losses
    plt.figure(figsize=(10, 6))
    plt.plot(ae_train_losses, label='Training Loss')
    plt.plot(ae_val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Autoencoder Training and Validation Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('autoencoder_loss.png')
    plt.close()

    # 2. Train and evaluate the Variational Autoencoder (VAE)
    print("\n=== Variational Autoencoder ===")

    # Initialize model and optimizer
    vae = VAE(latent_dim=latent_dim).to(device)
    vae_optimizer = optim.Adam(vae.parameters(), lr=1e-3)

    # Train the model
    vae, vae_train_losses, vae_val_losses = train_model(
        vae, train_loader, val_loader, vae_optimizer,
        num_epochs=20, is_vae=True, early_stopping_patience=5,
        checkpoint_path='checkpoints/vae.pt'
    )

    # Test the model
    vae_test_loss = test_model(vae, test_loader, is_vae=True)

    # Visualize latent space
    vae_latent_plot = visualize_latent_space(vae, test_loader, is_vae=True)
    vae_latent_plot.savefig('vae_latent_space.png')
    plt.close()

    # Visualize reconstructions
    vae_recon_plot = visualize_reconstructions(vae, test_loader, is_vae=True)
    vae_recon_plot.savefig('vae_reconstructions.png')
    plt.close()

    # Plot training and validation losses
    plt.figure(figsize=(10, 6))
    plt.plot(vae_train_losses, label='Training Loss')
    plt.plot(vae_val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('VAE Training and Validation Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('vae_loss.png')
    plt.close()

    print("\nExecution completed. Results saved as PNG files.")

Using device: cuda
Dataset sizes - Train: 48000, Validation: 6000, Test: 6000

=== Simple Autoencoder ===
Epoch: 1/20, Batch: 0/750, Loss: 181.541565
Epoch: 1/20, Batch: 100/750, Loss: 44.926300
Epoch: 1/20, Batch: 200/750, Loss: 27.398445
Epoch: 1/20, Batch: 300/750, Loss: 23.763723
Epoch: 1/20, Batch: 400/750, Loss: 20.586733
Epoch: 1/20, Batch: 500/750, Loss: 17.804129
Epoch: 1/20, Batch: 600/750, Loss: 16.570572
Epoch: 1/20, Batch: 700/750, Loss: 15.637486
Epoch: 1/20, Train Loss: 27.332246, Validation Loss: 15.015395
Checkpoint saved to checkpoints/autoencoder.pt
Epoch: 2/20, Batch: 0/750, Loss: 14.799379
Epoch: 2/20, Batch: 100/750, Loss: 13.558154
Epoch: 2/20, Batch: 200/750, Loss: 12.250477
Epoch: 2/20, Batch: 300/750, Loss: 11.865236
Epoch: 2/20, Batch: 400/750, Loss: 10.468414
Epoch: 2/20, Batch: 500/750, Loss: 10.741356
Epoch: 2/20, Batch: 600/750, Loss: 11.122505
Epoch: 2/20, Batch: 700/750, Loss: 11.342216
Epoch: 2/20, Train Loss: 12.279551, Validation Loss: 10.431970
Chec